# Serve + Test the E4B adapter on Colab T4 (HF nf4 — full fidelity)

Serves your trained E4B LoRA at the SAME nf4 4-bit precision it was trained on (no GGUF
quant degradation), on a free Colab **T4 (16GB)** which fits E4B 4-bit with room. This is
the truest read of model quality — q4_0 GGUF would confound it (and we've seen q4_0
fabricate numbers).

**Two things:** Cell 7 runs my probe prompts (you read the raw behavior); Cell 8 serves the
web UI via an ngrok tunnel so YOU can chat-test in the browser.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Add Colab Secrets:
`HF_TOKEN` (HF read) and `NGROK_TOKEN` (free from ngrok.com → authtoken). Put your
downloaded `best/` adapter folder in Google Drive (e.g. `MyDrive/chess_adapter/best`).

In [ ]:
# Cell 1 - config
import os
BASE_REPO = "unsloth/gemma-4-E4B-it"   # the base your adapter was trained on
REPO_URL  = "https://github.com/RyanDev1st/llm-and-engine.git"
BRANCH    = "feat/chess-coach-sft"

if os.path.exists("A:/Download/llm_tool_calling_research_workspace"):
    WORKDIR = "A:/Download/llm_tool_calling_research_workspace"
    ADAPTER_DIR = "A:/Download/gemma4_chess_e4b_kaggle (1)"
else:
    WORKDIR = "/content/llm-and-engine"
    ADAPTER_DIR = "/content/adapter"

BASE_DIR  = f"{WORKDIR}/src/llm/models/gemma4_e4b"
print("base:", BASE_REPO, "| adapter:", ADAPTER_DIR)

In [ ]:
# Cell 2 - GPU check (want a T4 16GB)
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.free","--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print("device:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 3 - clone repo
import os, subprocess
env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO_URL,WORKDIR], check=True, env=env)
else:
    subprocess.run(["git","-C",WORKDIR,"pull","--ff-only"], check=True, env=env)
subprocess.run(["git","-C",WORKDIR,"log","-1","--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])

In [ ]:
# Cell 4 - deps (serve path + ngrok + stockfish for full chess tools)
import subprocess, sys, shutil
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)
pip("-U","transformers","accelerate","peft","bitsandbytes","sentencepiece","protobuf","python-chess","pyngrok")
if shutil.which("apt-get"):
    subprocess.run(["apt-get","-qq","install","-y","stockfish"])  # chess tools (optional)
import transformers, peft, bitsandbytes
print("transformers",transformers.__version__,"| peft",peft.__version__,"| bnb",bitsandbytes.__version__)

In [ ]:
# Cell 5 - HF login + download the E4B base (~9GB; once)
import os
from huggingface_hub import login, snapshot_download
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception:
    login(os.environ["HF_TOKEN"])
snapshot_download(repo_id=BASE_REPO, local_dir=BASE_DIR,
                  allow_patterns=["*.json","*.safetensors","*.model","*.txt","tokenizer*"])
print("base at", BASE_DIR, "->", sorted(os.listdir(BASE_DIR))[:8])

In [ ]:
# Cell 6 - download the adapter from Hugging Face (FORMAT-FIXED run = ckpt-v4)
# v4 run: all-linear + FORMAT_WEIGHT=8.0 on the control tags + the serve-side </skill>
# stop fix. best/ = lowest-val checkpoint. val is NOT the judge here — Cell 7 (train-row
# free-gen probe) and Cell 7c (routing confusion matrix + precision) are.
#   TAG='best'       -> lowest-val adapter (PROBE/SERVE THIS)
#   TAG='checkpoint' -> latest step (only for resume)
import os
from huggingface_hub import snapshot_download

ADAPTER_REPO = "RyanDev1st/gemma4-chesscoach-ckpt-v4"   # current run (v3/v2/v1 superseded)
TAG = "best"                                            # lowest-val checkpoint

if not os.path.exists("A:/Download/llm_tool_calling_research_workspace"):
    print(f"Downloading adapter from HF: {ADAPTER_REPO} :: {TAG}/")
    snapshot_download(repo_id=ADAPTER_REPO, local_dir="/content/hf_adapter_v4",
                      allow_patterns=[f"{TAG}/*"])
    ADAPTER_DIR = f"/content/hf_adapter_v4/{TAG}"

need = ("adapter_model.safetensors", "adapter_config.json")
ok = all(os.path.exists(f"{ADAPTER_DIR}/{f}") for f in need)
if os.path.exists(ADAPTER_DIR):
    print("adapter at", ADAPTER_DIR, "->", os.listdir(ADAPTER_DIR))
    # sanity: all-linear adapter MUST have MLP modules (gate/up/down), not just attn
    import json as _j
    cfg = _j.loads(open(f"{ADAPTER_DIR}/adapter_config.json").read())
    print("target_modules:", cfg.get("target_modules"), "| r:", cfg.get("r"),
          "| alpha:", cfg.get("lora_alpha"), "| dropout:", cfg.get("lora_dropout"))
else:
    print("adapter at", ADAPTER_DIR, "-> NOT FOUND")
assert ok, f"adapter not found at {ADAPTER_DIR}"

In [ ]:
# Cell 6b - (alt) upload the adapter instead of Drive. Zip best/ locally, upload here.
# from google.colab import files; import zipfile, io, os
# up = files.upload()                       # pick best.zip
# name = next(iter(up))
# ADAPTER_DIR = "/content/best"
# with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall("/content")
# print(os.listdir(ADAPTER_DIR))

In [ ]:
# Cell 7 - PROBE (DECISIVE). Two tests against the step-400 all-linear adapter:
#  (A) TRAIN-ROW REPRODUCTION — feed the EXACT training-row system prompt (per-row skills
#      index + tool manifest, same as eval_routing._system) and the row's user turn, then
#      compare to the gold assistant turn. If the model can't emit the format on a row it
#      TRAINED on, format-emission is broken (this is what killed attn-only). Auto-classified:
#        OK     = emits <skill>/<tool>, NO foreign tags
#        SOUP   = foreign tags (<action>/<thought>/</goals>/<unused..>) = attn-only failure
#        NO-EMIT= no <skill>/<tool> at all
#  (B) GENERIC SERVE — real-world prompts via the default served catalog (what a user sees).
import os, sys, subprocess, time, json, re, gzip, urllib.request
os.environ['CHESS_HF_BASE'] = BASE_DIR
SVC = 'http://127.0.0.1:7861'

def _health():
    try: return json.load(urllib.request.urlopen(SVC+'/health', timeout=5)).get('ok')
    except Exception: return False

# adapter changed (now v2/best) -> force a fresh load; a reused service would hold the old one
subprocess.run(['pkill','-f','backend.model_server']); time.sleep(3)

def ensure_service():
    if _health():
        print('model service already UP - reusing'); return
    env={**os.environ,'PYTHONPATH':f'{WORKDIR}/src/llm','CHESS_HF_ADAPTER':ADAPTER_DIR,'CHESS_HF_BASE':BASE_DIR}
    log=open('/content/model_server.log','w')
    globals()['_MSVC']=subprocess.Popen([sys.executable,'-u','-m','backend.model_server',ADAPTER_DIR],
                                        cwd=f'{WORKDIR}/src/llm',env=env,stdout=log,stderr=subprocess.STDOUT)
    print('starting model service with', ADAPTER_DIR, '(loads ONCE, ~1-2 min)...')
    for _ in range(180):
        time.sleep(2)
        if globals()['_MSVC'].poll() is not None:
            print(open('/content/model_server.log').read()[-3000:]); raise RuntimeError('service crashed')
        if _health(): print('model service UP'); return
    print(open('/content/model_server.log').read()[-3000:]); raise RuntimeError('service did not come up')

ensure_service()
sys.path.insert(0, f'{WORKDIR}/src/llm')
from backend.inference import build_system_prompt
from llm_training.system_prompt import build_system   # same prompt builder training used

def _gen(sysp, user, mx=140, stop=('</tool>','</skill>')):
    body=json.dumps({'messages':[{'role':'system','content':sysp},{'role':'user','content':user}],
                     'max_new_tokens':mx,'stop':list(stop),'use_adapter':True}).encode()
    req=urllib.request.Request(SVC+'/generate',data=body,headers={'content-type':'application/json'})
    return json.load(urllib.request.urlopen(req,timeout=180))['text']

# ---------- (A) TRAIN-ROW REPRODUCTION ----------
CONTRACT={'think','/think','goal','/goal','plan','/plan','skill','/skill','tool','/tool'}
def classify(out):
    tags=re.findall(r'</?([a-zA-Z_][\w]*)', out)
    foreign=sorted({t for t in tags if t not in CONTRACT})
    emits=('<skill>' in out) or ('<tool>' in out)
    return ('OK' if emits and not foreign else 'SOUP' if foreign else 'NO-EMIT'), foreign

rows=[]
p=f"{WORKDIR}/data/sft/v1_2_train.jsonl"
if not os.path.exists(p): p+='.gz'
op=gzip.open if p.endswith('.gz') else open
with op(p,'rt',encoding='utf-8') as f:
    for i,line in enumerate(f):
        if i>=800: break
        line=line.strip()
        if line: rows.append(json.loads(line))
def pick(mode,n): return [r for r in rows if r.get('reasoning_mode')==mode][:n]
samples=pick('fast',2)+pick('think',2)+pick('auto',2)
print('\n========== (A) TRAIN-ROW REPRODUCTION (step-400 all-linear) ==========')
tally={'OK':0,'SOUP':0,'NO-EMIT':0}
for r in samples:
    sysp=build_system(r['skills_index'],r['tool_manifest'],r.get('plugin_context',{}),
                      reasoning_mode=r.get('reasoning_mode',''))
    user=next(m['content'] for m in r['messages'] if m['role']=='user')
    gold=next(m['content'] for m in r['messages'] if m['role']=='assistant')
    out=_gen(sysp,user); v,foreign=classify(out); tally[v]+=1
    print('='*64); print(f"[{r.get('reasoning_mode')}] slice={r.get('slice')} -> {v}   foreign={foreign}")
    print('USER :',user[:90]); print('GOLD :',gold[:150]); print('MODEL:',out[:300])
print('\nTRAIN-ROW TALLY:',tally,
      '  => verdict:', 'WIN (ship step-400)' if tally['OK']>=4 else
                       'SOUP=all-linear did NOT fix gen; data problem' if tally['SOUP']>=3 else 'MIXED')

# ---------- (B) GENERIC SERVE (real-world) ----------
print('\n========== (B) GENERIC SERVE ==========')
def step(prompt, mode):
    out=_gen(build_system_prompt(reasoning_mode=mode), prompt, mx=220)
    print(f'### [{mode}] {prompt}'); print(out); print('-'*70)
step('I scored 70, 10, and 8 - is my average above 85?', 'think')
step('What is the best move here?', 'auto')
step('hey, what can you do?', 'fast')


In [ ]:
# Cell 7a - PLAN-MODE behavioral probe: does the v4 adapter actually emit <goal>/<plan> on
# the plan signal? The earlier probes only sampled fast/think/auto, so plan was NEVER
# exercised — but the corpus DOES train it: V1_S_compound_plan + V1_T_audited_plan, ~2.2k
# train rows (3%), every one a concrete <goal> + <plan> checklist. Reuses the Cell 7 service
# + helpers (_gen, build_system_prompt). Expect a <goal> then a <plan> of '- [ ] action
# (binding)' boxes (the serve plan-gate + UI render exactly these), then tool steps.
PLAN_PROMPTS = [
    'two things: first tell me who is winning, then lay out a plan to improve my position',
    'check my average of 86, 82, 76 is over 80, and also whether $38.20 split 3 ways is under $15 each',
    'help me with two things: review my last move, then suggest the best continuation',
]
print('\n========== PLAN MODE (does it emit <goal>/<plan>?) ==========')
ok = 0
for p in PLAN_PROMPTS:
    out = _gen(build_system_prompt(reasoning_mode='plan'), p, mx=220, stop=('</plan>', '</tool>', '</skill>'))
    has_plan = '<plan>' in out
    ok += has_plan
    print(f"### [plan] emits<plan>={has_plan} <goal>={'<goal>' in out} | {p}")
    print(out); print('-' * 70)
print(f'PLAN tally: {ok}/{len(PLAN_PROMPTS)} emitted a <plan> checklist',
      '=> plan mode WORKS' if ok >= 2 else
      '=> plan rarely emitted — inspect (corpus trains it; may need more plan rows/steps or it is mode-mixing)')

In [ ]:
# Cell 7b - GENERALIZATION EVAL (run AFTER the full run, with a WORKING adapter loaded).
# Answers the two product questions with evidence, not assertion:
#  (A) STRESS — casual/messy phrasing that's in NO training row (slang, typos, multi-intent).
#      Tests: does it stay in-format + route sensibly when the input drifts? (base Gemma
#      carries language understanding; we check the harness holds.)
#  (B) UNSEEN SKILLS/TOOLS — a catalog of skills+tools from domains ABSENT from training
#      (tax, music, cooking, unit/currency). Tests the CORE product claim: route by reading
#      the in-context DESCRIPTION (not memory), fill an unseen tool's args from its schema,
#      and DECLINE when nothing fits (V1_B). Reuses the service from Cell 7 (SVC must be UP).
import json, re, urllib.request
from llm_training.system_prompt import build_system
from backend.inference import build_system_prompt
SVC = 'http://127.0.0.1:7861'

def _g(sysp, user, mx=140):
    body=json.dumps({'messages':[{'role':'system','content':sysp},{'role':'user','content':user}],
                     'max_new_tokens':mx,'stop':['</tool>','</skill>'],'use_adapter':True}).encode()
    req=urllib.request.Request(SVC+'/generate',data=body,headers={'content-type':'application/json'})
    return json.load(urllib.request.urlopen(req,timeout=180))['text']

CONTRACT={'think','/think','goal','/goal','plan','/plan','skill','/skill','tool','/tool'}
def emitted(out):
    ms=re.search(r'<skill>\s*([\w-]+)', out); mt=re.search(r'<tool>\s*([\w-]+)((?:\s+[\w-]+=\S+)*)', out)
    if ms: return ('skill', ms.group(1), '')
    if mt: return ('tool', mt.group(1), mt.group(2).strip())
    return (None, None, '')
def fmt_ok(out):
    return not [t for t in re.findall(r'</?([a-zA-Z_][\w]*)', out) if t not in CONTRACT]

# ---------- (A) HELD-OUT CASUAL STRESS (served chess catalog) ----------
print('\n========== (A) STRESS: messy phrasing not in training ==========')
STRESS=[('ok so like what do i even do here lol','auto'),('bruh is my position cooked??','auto'),
        ('yo can u check if im bout to get mated','think'),('explain like im 5 why my knight good','auto'),
        ('waht is teh best mvoe rn','auto'),('whats this opening even called','auto'),
        ('help i keep hanging my queen every game','think'),('is it my turn?? im confused','fast'),
        ('i wanna attack but idk how','auto'),('can we just play, push e4','fast')]
sf={'OK':0,'SOUP':0,'NO-EMIT':0}
for p,m in STRESS:
    out=_g(build_system_prompt(reasoning_mode=m), p)
    k,name,_=emitted(out); v='SOUP' if not fmt_ok(out) else ('OK' if k else 'NO-EMIT'); sf[v]+=1
    print(f"[{m}] {v:7} route={k}:{name}  | {p}\n   -> {out[:150]}")
print('STRESS format tally:', sf)

# ---------- (B) UNSEEN SKILL/TOOL ROUTING (catalog from domains NOT in training) ----------
print('\n========== (B) UNSEEN skills/tools (zero-shot routing) ==========')
SK=[{'name':'tax-filing-helper','description':'Use when the user asks about filing taxes, deductions, tax forms, brackets, or filing deadlines.','plugin':'finance'},
    {'name':'guitar-tab-reader','description':'Use when the user wants to read, interpret, or play guitar tablature, chords, or fingering.','plugin':'music'},
    {'name':'recipe-scaler','description':'Use when the user wants to scale a recipe up or down for a different number of servings.','plugin':'cooking'}]
TL=[{'name':'convert_units','description':'Convert a numeric value between measurement units.','args':{'value':'required','from_unit':'required','to_unit':'required'},'applies_when':'always'},
    {'name':'currency_rate','description':'Get the current exchange rate between two currencies.','args':{'base':'required','quote':'required'},'applies_when':'always'}]
PC={'installed':['finance','music','cooking'],'enabled':['finance','music','cooking'],'marketplace':[]}
CASES=[
 {'p':'I need to file my taxes this year, where do I even start?','m':'auto','kind':'skill','t':'tax-filing-helper'},
 {'p':'can you help me read this guitar tab?','m':'auto','kind':'skill','t':'guitar-tab-reader'},
 {'p':'scale my cookie recipe from 12 to 30 servings','m':'auto','kind':'skill','t':'recipe-scaler'},
 {'p':'convert 5 miles into kilometers','m':'auto','kind':'tool','t':'convert_units','args':['value','from_unit','to_unit']},
 {'p':'what is the USD to EUR exchange rate right now?','m':'auto','kind':'tool','t':'currency_rate','args':['base','quote']},
 {'p':'what is the capital of France?','m':'auto','kind':'none','t':None}]  # nothing fits -> must DECLINE
tally={'CORRECT':0,'MIS-ROUTE':0,'NO-EMIT':0}
for c in CASES:
    out=_g(build_system(SK,TL,PC,reasoning_mode=c['m']), c['p'])
    k,name,argstr=emitted(out)
    if c['kind']=='none':
        v='CORRECT' if k is None else 'MIS-ROUTE'
    elif k is None:
        v='NO-EMIT'
    elif k==c['kind'] and name==c['t']:
        v='CORRECT'
        if c.get('args'):
            miss=[a for a in c['args'] if a+'=' not in argstr]
            if miss: v=f'CORRECT(name) BAD-ARGS missing={miss}'
    else:
        v='MIS-ROUTE'
    tally[v.split()[0]] = tally.get(v.split()[0],0)+1
    print(f"{v:10} want={c['kind']}:{c['t']} got={k}:{name} {argstr}  | {c['p']}\n   -> {out[:160]}")
print('\nUNSEEN routing tally:', tally,
      '=> PRODUCT CLAIM HOLDS' if tally.get('CORRECT',0)>=5 else '=> zero-shot routing weak — inspect')


In [ ]:
# Cell 7c - CONFUSION MATRIX + PRECISION from the v4 adapter (for the slides).
# REUSES the Cell 7 model service over HTTP (RemoteModel) -> the adapter is loaded ONCE
# and shared; this does NOT load a second E4B, so it cannot OOM the T4 (the failure mode
# from loading HFModel twice). Routing is scored as a 3-class verb decision on the model's
# FIRST action vs the gold first action over the validation set: skill / tool / none. Prints
# the 3x3 matrix (rows=gold, cols=pred) + per-class precision/recall/F1 + exact-name accuracy,
# and writes docs/findings/<date>-routing-confusion.{md,png}.
import os, sys, subprocess
from datetime import date
from pathlib import Path

ensure_service()                       # the ONE model service must be up (reuses it; no reload)
PER_SLICE = 20                         # rows per slice (~10-15 min). Set 0 for the FULL val set (~60 min).
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
print(f'running routing confusion eval against the live service (reusing the loaded v4 adapter), '
      f'per_slice={PER_SLICE} ...')
subprocess.run([sys.executable, '-m', 'llm_training.eval_confusion',
                '--server', SVC, '--per-slice', str(PER_SLICE)],
               cwd=f'{WORKDIR}/src/llm', env=env, check=True)

# show the saved confusion-matrix PNG inline (for the deck)
png = Path(f'{WORKDIR}/docs/findings/{date.today():%Y-%m-%d}-routing-confusion.png')
if png.exists():
    from IPython.display import Image, display
    display(Image(filename=str(png)))
    print('saved:', png)
else:
    print('(no PNG — matplotlib may be unavailable; the text matrix above still stands)')

In [ ]:
# Cell 8 - SERVE the web UI (reuses the SAME model service from Cell 7; ngrok tunnel).
# Only the weightless APP restarts here - the model is NOT reloaded (no OOM).
import os, sys, subprocess, time
ensure_service()                      # reuse the one model; starts it only if not already up
env={**os.environ,'PYTHONPATH':f'{WORKDIR}/src/llm','CHESS_HF_ADAPTER':ADAPTER_DIR,'CHESS_HF_BASE':BASE_DIR,
     'CHESS_MODEL_SERVER':SVC,'CHESS_HOST':'127.0.0.1','CHESS_PORT':'7860'}
subprocess.run(['pkill','-f','backend.server']); time.sleep(2)   # restart app only (cheap, weightless)
globals()['_APP']=subprocess.Popen([sys.executable,'-u','-m','backend.server'],cwd=f'{WORKDIR}/src/llm',
                  env=env,stdout=open('/content/app_server.log','w'),stderr=subprocess.STDOUT)
time.sleep(5)
from pyngrok import ngrok
ngrok.kill()                          # drop old tunnels (free tier allows 1)
try:
    from google.colab import userdata; ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
except Exception:
    ngrok.set_auth_token(os.environ['NGROK_TOKEN'])
url = ngrok.connect(7860)
print('==============================================')
print('  OPEN THIS IN YOUR BROWSER TO CHAT-TEST:')
print('  ', url)
print('==============================================')
print('--- app server log ---')
print(open('/content/app_server.log').read()[-1500:])
